# test_nodes

用于直接验证 route 与四类执行节点。

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (candidate / 'src' / 'task_router_graph').exists()
)
SRC_ROOT = PROJECT_ROOT / 'src'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from task_router_graph.nodes import (
    route_node,
    normal_node,
    functest_node,
    accutest_node,
    perftest_node,
    update_node,
)
from task_router_graph.schema import Environment
from task_router_graph.graph import TaskRouterGraph

print('模块加载完成')
print('当前 Python：', sys.executable)
print('已检测 API_KEY_Qwen：', bool(os.getenv('API_KEY_Qwen')))


In [10]:
# show_environment 已内置在 Environment 类中，直接调用 env.show_environment(...)


In [ ]:
graph = TaskRouterGraph(config_path='configs/graph.yaml')
env = Environment()
run_root = (graph.root / graph.config['paths']['run_root']).resolve()

task, trace = route_node(
    llm=graph._llm,
    controller_system=graph._controller_system,
    controller_skills_index=graph._controller_skills_index,
    environment=env,
    user_input='请帮我做一次 anthropic_ver_1 的功能测试',
    workspace_root=graph.root,
    run_root=run_root,
    max_steps=graph._max_controller_steps,
)

print('路由完成')
print('任务类型：', task.type)
print('动作条数：', len(trace))
print('最后动作：', trace[-1].action_kind)


In [12]:
print(env.show_environment())


=== Environment ===
updated_at: 2026-04-07T05:06:54.863542+00:00
rounds: 0
round_details: <empty>


In [ ]:
execute_map = {
    'normal': lambda t: normal_node(
        llm=graph._llm,
        normal_system=graph._normal_system,
        normal_skills_index=graph._normal_skills_index,
        environment=env,
        task=t,
    ),
    'functest': lambda t: functest_node(task=t),
    'accutest': lambda t: accutest_node(task=t),
    'perftest': lambda t: perftest_node(task=t),
}

task2, reply = execute_map[task.type](task)
env = update_node(env, '请帮我做一次 anthropic_ver_1 的功能测试', trace, task2, reply)

print('执行与回写完成')
print('任务状态：', task2.status)
print('任务结果：', task2.result)
print('回复内容：', reply)
print('环境轮次：', len(env.build_observation_view(round_limit=None)))


In [14]:
print(env.show_environment(show_trace=True))


=== Environment ===
updated_at: 2026-04-07T05:06:54.863542+00:00
rounds: 1

[Round 1]
user_input: 请帮我做一次 anthropic_ver_1 的功能测试
task_type: functest
task_status: done
task_result: functest 已完成（示例执行）：针对 anthropic_ver_1 执行功能测试，重点验证核心功能行为与断言结果
reply: [functest] 已完成（示例断言）
trace_len: 2
  trace#1: kind=observe
    tool=read
    args={'path': 'src/task_router_graph/skills/controller/functest-task.md'}
    reason=USER_INPUT 明确请求 anthropic_ver_1 的功能测试，属于 functest 场景 A（对象明确），根据规则应优先读取 functest reference 以确认 task_content 生成条件
    observation=﻿# Functest Task Reference  ## 1. 定位  - `functest` 用于生成“功能测试目标（target）”。 - controller 在这一层负责确定：本轮测什么、围绕什么测、重点朝哪个方向测。 - controller 不负责在这一层补齐完整执行配置。  ## 2. 常见场景分类  ### 场景 A：明确对象的直接功能测试  示例：  - 请帮我做一次 an...
  trace#2: kind=generate_task
    tool=None
    args={}
    reason=USER_INPUT 明确指定测试对象为 anthropic_ver_1，属于 functest 场景 A（对象明确），且 functest-task.md 确认无需额外外部事实即可生成 task_content；推荐写法已直接匹配该场景
    task_type=functest
    task_content=针对 anthropic_ver_1 执行功能测试，重点验证核心功能

In [ ]:
trace_view = env.build_observation_view(include_trace=True)
trace_view[0]['controller_trace']
